In [74]:

import json
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import numpy as np
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Concatenate
from tensorflow.keras.models import Model
from keras.optimizers import SGD
from keras.losses import binary_crossentropy

In [48]:
folder_path = '/Users/karinamehta/Desktop/speeches'
all_speeches = []
for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)
        
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            all_speeches.append(data)

print(f"Done {len(all_speeches)}")

Done 1052


In [34]:
def clean_speech_noise(text):
    """
    input: (str) speech
    output: (str) formatted speech
    Cleans text from noise and puts everything in lowercase
    """
    
    text = text.replace('<br />', ' ')
    text = text.replace('*', '')
    text = re.sub(r'[\n\r]+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    
    return text

In [36]:
party_labels = {
    "George Washington": 2,      # Independent
    "John Adams": 3,             # Federalist
    "Thomas Jefferson": 4,       # Democratic-Republican
    "James Madison": 4,          # Democratic-Republican
    "James Monroe": 4,           # Democratic-Republican
    "John Quincy Adams": 4,      # Democratic-Republican
    "Andrew Jackson": 0,         # Democrat
    "Martin Van Buren": 0,       # Democrat
    "William Henry Harrison": 5, # Whig
    "John Tyler": 5,             # Whig
    "James K. Polk": 0,          # Democrat
    "Zachary Taylor": 5,         # Whig
    "Millard Fillmore": 5,       # Whig
    "Franklin Pierce": 0,        # Democrat
    "James Buchanan": 0,         # Democrat
    "Abraham Lincoln": 1,        # Republican
    "Andrew Johnson": 6,         # National Union
    "Ulysses S. Grant": 1,       # Republican
    "Rutherford B. Hayes": 1,    # Republican
    "James A. Garfield": 1,      # Republican
    "Chester A. Arthur": 1,      # Republican
    "Grover Cleveland": 0,       # Democrat
    "Benjamin Harrison": 1,      # Republican
    "William McKinley": 1,       # Republican
    "Theodore Roosevelt": 1,     # Republican
    "William Howard Taft": 1,    # Republican
    "Woodrow Wilson": 0,         # Democrat
    "Warren G. Harding": 1,      # Republican
    "Calvin Coolidge": 1,        # Republican
    "Herbert Hoover": 1,         # Republican
    "Franklin D. Roosevelt": 0,  # Democrat
    "Harry S. Truman": 0,        # Democrat
    "Dwight D. Eisenhower": 1,   # Republican
    "John F. Kennedy": 0,        # Democrat
    "Lyndon B. Johnson": 0,      # Democrat
    "Richard Nixon": 1,          # Republican
    "Gerald Ford": 1,            # Republican
    "Jimmy Carter": 0,           # Democrat
    "Ronald Reagan": 1,          # Republican
    "George H.W. Bush": 1,       # Republican
    "Bill Clinton": 0,           # Democrat
    "George W. Bush": 1,         # Republican
    "Barack Obama": 0,           # Democrat
    "Donald Trump": 1,           # Republican
    "Joe Biden": 0               # Democrat
}
x = [] #txt
y = [] #labels

for speech in all_speeches:
    president = speech.get('president')
    transcript = speech.get('transcript')
    clean_transcript = clean_speech_noise(transcript)
    if president in party_labels:
        x.append(clean_transcript)
        y.append(party_labels[president])

In [45]:
max_vocab = 10000 #max words
tokenizer = Tokenizer(num_words=max_vocab)
tokenizer.fit_on_texts(x) 
X_numeric = tokenizer.texts_to_sequences(x)

#padding
max_speech_length = 500  #500 words
X_padded = pad_sequences(X_numeric, maxlen=max_speech_length, padding='post', truncating='post')
y_array = np.array(y)

# 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, 
    y_array, 
    test_size=0.2, 
    random_state=42
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (794, 500)
Testing data shape: (199, 500)


In [92]:
inpx = Input(shape=(max_speech_length,), name="speech_input")

x = Embedding(input_dim=max_vocab, output_dim=embedding_dim)(inpx)
x = Conv1D(filters=128, kernel_size=5, activation='relu')(x)
x = GlobalMaxPooling1D()(x)
hid = Dense(64, activation='relu')(x) 
hid = Dropout(0.5)(hid)
out_layer = Dense(1, activation='sigmoid')(hid)

model = Model(inputs=[inpx], outputs=[out_layer])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [94]:
model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

In [98]:
#training, its terrible for now
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.3715 - loss: 0.3215 - val_accuracy: 0.3819 - val_loss: 0.0468
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.3602 - loss: -0.3136 - val_accuracy: 0.3819 - val_loss: -0.3488
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.3602 - loss: -0.8942 - val_accuracy: 0.3819 - val_loss: -1.3315
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.3602 - loss: -2.4261 - val_accuracy: 0.3819 - val_loss: -4.1499
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.3602 - loss: -7.0218 - val_accuracy: 0.3819 - val_loss: -12.2688
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.3602 - loss: -23.1943 - val_accuracy: 0.3819 - val_loss: -32.7898
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.3602 - loss: -62.3438 - val_accuracy: 0.3819 - val_loss: -91.6994
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.3678 - loss: -145.4523 - val_